# Cross-table checks
            Goal: check that the tables match each other. Issues are only found and logged here
            (docs/data_issues.md); they are fixed later in PostgreSQL.

## 0. Setup

In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv", parse_dates = ['signup_date'], dtype = {'birth_year': 'Int64'})
products = pd.read_csv("../data/raw/products.csv", parse_dates = ['launch_date', 'discontinued_date'])
orders = pd.read_csv("../data/raw/orders.csv", parse_dates = ['order_ts', 'promised_date', 'delivered_date'], dtype = {'session_id' : 'Int64'})
orders.insert(4, "order_date_local", orders["order_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
order_items = pd.read_csv("../data/raw/order_items.csv", parse_dates = ['return_date'])
user_events = pd.read_csv("../data/raw/user_events.csv", parse_dates = ['event_ts'], dtype = {'customer_id': 'Int64', 'product_id': 'Int64'})
user_events.insert(6, "event_date_local", user_events["event_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
marketing_spend = pd.read_csv("../data/raw/marketing_spend.csv", parse_dates = ['spend_date'], dtype = {'platform_conversions' : 'Int64'})

## Cross-table questions

        ### A. Keys – does every ID have a match in the other table?
        - A1. order_items.order_id → orders.order_id
        - A2. orders.order_id → order_items.order_id
        - A3. order_items.product_id → products.product_id
        - A4. orders.customer_id → customers.customer_id
        - A5. orders.session_id → user_events.session_id
        - A6. user_events.product_id → products.product_id
        
        ### B. Same thing in two tables – do they match?
        - B1. Does order_total = sum of order lines + shipping_fee?
        - B2. Do large quantities (DQ-12) match order_total?
        - B3. Why do purchase events not match orders (DQ-15)?
        
        ### C. Time order – are dates in a logical order?
        - C1. Were products sold before launch_date or after discontinued_date?
        - C2. Is return_date after delivered_date?
        
        ### D. Status
        - D1. Are returned lines only in completed orders?

## A. Keys

### A1. Does every order_id in order_items exist in orders?

In [4]:
order_items_orders = order_items.merge(orders, how = "left_anti", on = "order_id")
print(len(order_items_orders))


0


**Answer:** Yes. 0 order lines without an order.

### A2. Does every order_id in orders exist in order_items?

In [5]:
orders_order_items = orders.merge(order_items, how = "left_anti", on = "order_id")
print(len(orders_order_items))

0


**Answer:** Yes. 0 orders without order lines.

### A3. Does every product_id in order_items exist in products?

In [7]:
order_items_products = order_items.merge(products, how = "left_anti", on = "product_id")
print(len(order_items_products))

0


**Answer:** Yes. 0 order lines with an unknown product.

### A4. Does every customer_id in orders exist in customers?

In [9]:
orders_customer_id = orders.merge(customers, how = "left_anti", on = "customer_id")
print(len(orders_customer_id))

0


**Answer:** Yes. 0 orders with unknown customer.

### A5. Does every session_id in orders exist in user_events?

In [12]:
orders_user_events = orders.dropna(subset=["session_id"]).merge(user_events, how = "left_anti", on = "session_id")
print(len(orders_user_events))

0


**Answer:** Yes. 584 orders have no session_id (expected, see data dictionary), so they were left out. All other orders have a session_id that exists in user_events.

### A6. Does every product_id in user_events exist in products?

In [14]:
user_events_products = user_events.dropna(subset=["product_id"]).merge(products, how = "left_anti", on = "product_id")
print(len(user_events_products))

0


**Answer:** Yes. 34616 user_events have no product_id (expected, see data dictionary), so they were left out. All other user_events have a product_id that exists in products.